# MOSE sample `0ab75563` — complete watermark/removal comparison

This notebook is one self-contained case study. It presents every video and result artifact committed for MOSE clip `0ab75563` across the unwatermarked baseline and the three requested watermark methods: **VideoSeal**, **Safe-Sora**, and **VideoShield**.

The experiment order is always:

```text
original → watermark embedding → watermarked video → ProPainter or DiffuEraser → detector/recovery
```

That ordering matters: the removal method attacks a video that was already watermarked. The unwatermarked baseline is shown separately so visual removal quality can be inspected without conflating it with watermark behavior.

> **Important capability result:** VideoShield cannot faithfully add its watermark to an existing MOSE video because it embeds into a diffusion generator's initial noise. The repository therefore contains a labelled negative control—not fabricated watermarked, ProPainter, DiffuEraser, mask, or IoU outputs—for this method/sample pair.

All media remains in the validated GitHub evidence bundle under `evidence/mose-task1/review50/media/0ab75563`. The notebook references those files instead of duplicating them.


## At-a-glance status

| Section | Status | What is available |
| --- | --- | --- |
| Unwatermarked baseline | Complete | Original, raw mask, ProPainter output, DiffuEraser output |
| VideoSeal | Complete | Watermarked input, both removal attacks, attack-specific effective GT masks, both derived predicted masks, localization metrics |
| Safe-Sora | Complete | Aligned sampled original, aligned mask, watermarked input, both removal attacks, two graphical-watermark recovery images, fidelity metrics |
| VideoShield | Not applicable for watermarking this MOSE clip | Shared original/mask plus an explicit negative-control and availability record |

**Source properties:** 104 frames, 960 × 448, 25 FPS, approximately 4.16 seconds.


## Metric guide

The methods do not expose the same kind of signal, so their numbers must not be treated as interchangeable.

| Term | Meaning | Direction and interpretation |
| --- | --- | --- |
| **Global pixel IoU** | `sum(predicted AND ground truth) / sum(predicted OR ground truth)` over every pixel in every frame. The ground truth is the effective mask actually consumed by the remover. | 1 is perfect overlap; 0 means no overlap. This is the primary spatial-localization number. |
| **Mean frame IoU** | IoU is calculated separately for every frame, then averaged. A frame where both masks are empty is conventionally scored 1. | Useful for finding temporal behavior, but it can look high when many frames contain no target. It is not interchangeable with global IoU. |
| **Minimum/maximum frame IoU** | Worst and best individual-frame IoU. | Shows the range across time; it does not describe total pixel overlap. |
| **Intersection pixels** | Number of pixels classified as tampered by both the predicted mask and effective ground truth. | Larger is not automatically better because object size varies; interpret together with union and IoU. |
| **Union pixels** | Number of pixels classified as tampered by either mask. | The denominator of global IoU. |
| **PSNR (dB)** | Logarithmic fidelity derived from mean squared error for 8-bit images. | Higher is more similar. Identical images have infinite PSNR. Values around 8 dB indicate a visually weak reconstruction; values above 40 dB indicate very small pixel differences. |
| **MSE** | Mean squared 8-bit pixel error. Large errors receive extra weight because they are squared. | Lower is better; 0 is identical. |
| **MAE** | Mean absolute 8-bit pixel error. | Lower is better; it is the average absolute change per channel value. |
| **Bit accuracy** | Fraction of binary watermark bits recovered correctly. | 1 is perfect. For a balanced binary watermark, about 0.5 is chance. |
| **Elapsed seconds** | Wall-clock runtime recorded for that stage on the experiment GPU. | Useful for this run, not a hardware-independent benchmark. |
| **Frames / FPS / dimensions** | Output length, playback rate, and width × height. | Used to prove that compared artifacts are aligned and decodable. |
| **Mask dilation iterations** | Number of times the selected-object mask was expanded before inpainting. | A configuration value, not a quality score. It gives the remover context around object boundaries. |
| **DDIM inversion steps** | Number of reverse-diffusion steps used by the VideoShield detector control. | A detector configuration value, not a quality score. |
| **SHA-256** | Content fingerprint stored in metadata. | An integrity check: matching hashes mean the exact recorded file is present. It is not a model metric. |
| **Complete / not applicable** | `complete` means the recorded stage produced and validated its expected artifacts. `not applicable` means the method cannot perform that experiment faithfully. | `Not applicable` is deliberately different from failure or a score of zero. |

### Mask terminology

- **Raw mask:** the original MOSE annotation chosen for object removal; white means “remove this region.”
- **Effective GT mask:** the mask after the remover's real resize, erosion, and/or dilation path. This is the correct IoU reference.
- **Predicted mask:** where a watermark detector says tampering occurred. For VideoSeal here it is a documented **derived 4 × 4 crop-grid proxy**, not a native pixelwise output from the released checkpoint.


## Notebook setup

After a fresh clone, first run `python3 scripts/prepare_mose_notebooks.py` from the repository root. That downloads and verifies all 48 notebook videos in one Git LFS operation. Then run this cell once: it locates the repository and defines self-contained inline video, image, and metadata viewers. Each media card also provides a download link. If a clone contains an LFS pointer instead of a real video, this cell reports the exact preparation command rather than showing a broken player.


In [ ]:
import base64
from pathlib import Path
import html
import json
from IPython.display import HTML, Image, Markdown, display

SAMPLE_ID = "0ab75563"
LFS_POINTER_PREFIX = b"version https://git-lfs.github.com/spec/v1"

def find_repo_root():
    for start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (start / "evidence/mose-task1/review50/manifest.json").is_file():
            return start
    raise FileNotFoundError("Run this notebook from inside the paper-reproductions checkout.")

REPO_ROOT = find_repo_root()
REVIEW_ROOT = REPO_ROOT / "evidence/mose-task1/review50"
SAMPLE_ROOT = REVIEW_ROOT / "media" / SAMPLE_ID
MANIFEST = json.loads((REVIEW_ROOT / "manifest.json").read_text())
SAMPLE = next(item for item in MANIFEST["samples"] if item["clip_id"] == SAMPLE_ID)

def artifact_data_url(source, mime_type):
    if not source.is_file():
        raise FileNotFoundError(source)
    payload = source.read_bytes()
    if payload.startswith(LFS_POINTER_PREFIX):
        raise RuntimeError(
            f"{source} is a Git LFS pointer, not the real media file. "
            "From the repository root run: python3 scripts/prepare_mose_notebooks.py"
        )
    return f"data:{mime_type};base64,{base64.b64encode(payload).decode('ascii')}"

def show_videos(items, columns=3):
    cards = []
    for title, relative, explanation in items:
        if relative is None:
            cards.append(f"<section class='media-card'><h4>{html.escape(title)}</h4><p><strong>Not applicable.</strong></p><p>{html.escape(explanation)}</p></section>")
            continue
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "video/mp4")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<video controls muted loop playsinline preload='metadata' src='{data_url}'></video>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this video</a></p>"
            "</section>"
        )
    display(HTML(
        f"<style>.media-grid{{display:grid;grid-template-columns:repeat({columns},minmax(220px,1fr));gap:16px}}"
        ".media-card{border:1px solid #d0d7de;border-radius:8px;padding:12px;background:#fff;color:#24292f}"
        ".media-card video{width:100%;background:#000}.media-card h4{margin:0 0 8px}.media-card p{font-size:13px}</style>"
        f"<div class='media-grid'>{''.join(cards)}</div>"
    ))

def show_images(items, columns=2):
    cards = []
    for title, relative, explanation in items:
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "image/png")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<img style='width:100%' src='{data_url}' alt='{html.escape(title)}'>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this image</a></p>"
            "</section>"
        )
    display(HTML(f"<div class='media-grid' style='grid-template-columns:repeat({columns},minmax(260px,1fr))'>{''.join(cards)}</div>"))

def show_metadata(paths):
    for relative in paths:
        path = REVIEW_ROOT / relative
        payload = json.loads(path.read_text())
        display(Markdown(f"#### `{relative}`"))
        print(json.dumps(payload, indent=2))

print(f"Loaded sample {SAMPLE_ID} from {SAMPLE_ROOT}")


## 1. Unwatermarked removal baseline

This section answers a visual question: **can each inpainting method remove the selected object when no watermark is involved?**

- The original is the full MOSE clip.
- The raw mask shows the selected object in white.
- ProPainter and DiffuEraser each receive that selected object, with their recorded mask preprocessing.
- These two outputs are context only; they are not watermark-robustness results.

Direct links: [original](../../evidence/mose-task1/review50/media/0ab75563/original.mp4) · [raw mask](../../evidence/mose-task1/review50/media/0ab75563/gt_mask.mp4) · [ProPainter baseline](../../evidence/mose-task1/review50/media/0ab75563/propainter.mp4) · [DiffuEraser baseline](../../evidence/mose-task1/review50/media/0ab75563/diffueraser.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/0ab75563/original.mp4', 'The source clip before watermarking or removal.'), ('Raw selected-object mask', 'media/0ab75563/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter, no watermark', 'media/0ab75563/propainter.mp4', 'Baseline ProPainter removal on the unwatermarked source.'), ('DiffuEraser, no watermark', 'media/0ab75563/diffueraser.mp4', 'Baseline DiffuEraser removal on the unwatermarked source.')])


## 2. VideoSeal → removal → derived localization

VideoSeal post-processes an existing video, so it can follow the required protocol exactly. It embeds a 256-bit message into all 104 frames. ProPainter and DiffuEraser then remove the selected object from that watermarked input.

The released checkpoint exposes a global watermark-presence detector rather than a native pixelwise tamper map. This experiment therefore derives a clearly labelled **4 × 4 crop-grid absence mask**. White predicted-mask pixels mean the crop proxy judged the watermark absent there. These masks and IoUs must not be described as native VideoSeal localization.

### Recorded VideoSeal metrics

| Metric | ProPainter attack | DiffuEraser attack |
| --- | ---: | ---: |
| Removal runtime | 43.633 s | 98.975 s |
| Frames | 104 | 104 |
| Output dimensions | 960 × 448 | 960 × 448 |
| Global pixel IoU | **0.0000** | **0.0000** |
| Mean frame IoU | 0.8365 | 0.8365 |
| Minimum frame IoU | 0.0000 | 0.0000 |
| Maximum frame IoU | 1.0000 | 1.0000 |
| Intersection pixels | 0 | 0 |
| Union pixels | 490,646 | 499,284 |

Embedding runtime was 3.231 s; localization runtime for both attacks together was 10.619 s. The detection threshold was `0.0`: positive detector score means watermark present, and a score at or below the threshold becomes predicted tampering.

### Interpretation

Both global IoUs are 0.0000. The derived detector predicted no pixels overlapping the actual removed region, so the crop-grid proxy did not spatially localize the attack in this sample. This does **not** by itself prove that the watermark was absent everywhere; it says the documented spatial proxy failed to overlap the effective ground truth. The non-zero mean frame IoU comes from frames where both the predicted and effective-GT masks are empty; those frames score 1 individually. Global IoU still correctly reports zero overlap across the complete clip.


In [ ]:
show_videos([('VideoSeal input original', 'media/0ab75563/original.mp4', 'The exact source that receives the VideoSeal message.'), ('VideoSeal watermarked', 'media/0ab75563/methods/videoseal/watermarked.mp4', 'The watermarked video before either removal attack.'), ('Raw source mask', 'media/0ab75563/gt_mask.mp4', 'Human-readable MOSE annotation before remover-specific preprocessing.'), ('ProPainter attacked video', 'media/0ab75563/methods/videoseal/propainter.mp4', 'ProPainter output after attacking the VideoSeal-watermarked input.'), ('ProPainter effective GT', 'media/0ab75563/methods/videoseal/effective-gt-mask-propainter.mp4', "Mask after ProPainter's resize and eight dilation iterations; this is its IoU reference."), ('ProPainter predicted mask', 'media/0ab75563/methods/videoseal/pred-mask-propainter.mp4', 'Derived 4 × 4 VideoSeal absence-grid prediction after ProPainter.'), ('DiffuEraser attacked video', 'media/0ab75563/methods/videoseal/diffueraser.mp4', 'DiffuEraser output after attacking the VideoSeal-watermarked input.'), ('DiffuEraser effective GT', 'media/0ab75563/methods/videoseal/effective-gt-mask-diffueraser.mp4', "Mask after DiffuEraser's erosion/dilation path; this is its IoU reference."), ('DiffuEraser predicted mask', 'media/0ab75563/methods/videoseal/pred-mask-diffueraser.mp4', 'Derived 4 × 4 VideoSeal absence-grid prediction after DiffuEraser.')])


### VideoSeal run records

The next cell prints the exact committed embedding, attack, and localization JSON. It includes per-frame IoUs and SHA-256 fingerprints in addition to the summarized values above.


In [ ]:
show_metadata(['media/0ab75563/methods/videoseal/diffueraser.json', 'media/0ab75563/methods/videoseal/embed.json', 'media/0ab75563/methods/videoseal/localization.json', 'media/0ab75563/methods/videoseal/propainter.json'])


## 3. Safe-Sora → removal → graphical-watermark recovery

Safe-Sora is evaluated under its released 16-frame, 512 × 320 protocol. It samples 16 aligned frames from the MOSE clip, passes them through the VAE path, embeds a **graphical watermark**, and then applies each remover. Its released method reconstructs a watermark image; it does not produce a spatial tamper mask. Therefore predicted-mask and IoU fields are truthfully **not applicable**.

DiffuEraser requires at least 22 frames, so the final frame and mask were repeated to 24 frames only during inference and the output was trimmed back to 16 genuine frames.

### Embedding fidelity

| Comparison | PSNR | MSE | MAE | Meaning |
| --- | ---: | ---: | ---: | --- |
| Selected input → VAE cover reconstruction | 35.6311 dB | 17.7816 | 2.9008 | Measures ordinary VAE reconstruction change before the watermark difference is isolated. |
| VAE cover → watermarked output | **43.3406 dB** | 3.0132 | 1.2898 | Measures the additional visible pixel change attributable to embedding. |

The embedding PSNR is high and the errors are low, meaning the watermarked video is pixelwise close to Safe-Sora's cover reconstruction. Embedding took 5.677 s.

### Removal and watermark-recovery metrics

| Metric | ProPainter attack | DiffuEraser attack |
| --- | ---: | ---: |
| Removal runtime | 8.323 s | 27.844 s |
| Reported output frames | 16 | 16 |
| Recovery PSNR | **7.8723 dB** | **8.3995 dB** |
| Recovery MSE | 10613.2044 | 9400.0013 |
| Recovery MAE | 92.5715 | 86.0763 |
| Spatial predicted mask / IoU | N/A | N/A |

Graphical recovery for both attacks took 1.249 s in total. DiffuEraser has the higher recovery PSNR by 0.5272 dB for this sample, but both recovery PSNR values are around 8 dB, indicating a weak reconstruction of the original graphical watermark. This is a watermark-recovery result—not a judgment of which inpainted video looks better.


In [ ]:
show_videos([('Safe-Sora sampled original', 'media/0ab75563/methods/safe-sora/sampled-original.mp4', 'The aligned 16-frame, 512 × 320 source used by Safe-Sora.'), ('Safe-Sora watermarked', 'media/0ab75563/methods/safe-sora/watermarked.mp4', 'The graphical-watermarked input before removal.'), ('Safe-Sora aligned mask', 'media/0ab75563/methods/safe-sora/gt-mask.mp4', 'The source removal mask sampled and resized to the same 16 frames.'), ('Safe-Sora + ProPainter', 'media/0ab75563/methods/safe-sora/propainter.mp4', 'ProPainter removal applied after Safe-Sora embedding.'), ('Safe-Sora + DiffuEraser', 'media/0ab75563/methods/safe-sora/diffueraser.mp4', 'DiffuEraser removal applied after Safe-Sora embedding; temporary padding is not shown.')])


### Recovered graphical watermarks

Each image contains the recovery visualization produced from the attacked video. These images are not object masks and must not be used to calculate IoU.


In [ ]:
show_images([('Recovery after ProPainter', 'media/0ab75563/methods/safe-sora/recovery-propainter.png', 'Recovery PSNR: 7.8723 dB.'), ('Recovery after DiffuEraser', 'media/0ab75563/methods/safe-sora/recovery-diffueraser.png', 'Recovery PSNR: 8.3995 dB.')])


### Safe-Sora run records

The next cell prints the committed embedding, removal, and recovery JSON, including sampled frame indices, the compatibility adapter, all fidelity values, and checksums.


In [ ]:
show_metadata(['media/0ab75563/methods/safe-sora/diffueraser.json', 'media/0ab75563/methods/safe-sora/embed.json', 'media/0ab75563/methods/safe-sora/propainter.json', 'media/0ab75563/methods/safe-sora/recovery.json'])


## 4. VideoShield capability record and negative control

VideoShield's watermark is injected into a diffusion model's initial generation noise. This MOSE clip already exists, so VideoShield cannot add its authentic watermark by post-processing the clip. Consequently there is no truthful VideoShield-watermarked video to send to ProPainter or DiffuEraser, and there can be no attack-specific predicted mask or IoU for this sample.

The repository still records an explicit control: 16 frames were sampled, resized to 256 × 256, inverted for 25 DDIM steps, and compared with a reference watermark **without embedding that watermark into the clip**.

| Metric/configuration | Value | Interpretation |
| --- | ---: | --- |
| Negative-control bit accuracy | **0.4805** (48.05%) | 1.95 percentage points from the 50% chance level. This is expected for an unwatermarked input and is not robustness evidence. |
| Sampled frames | 16 | Evenly selected control frames from the source clip. |
| Detector resize | 256 × 256 | Resolution used by the control detector. |
| DDIM inversion steps | 25 | Reverse-diffusion configuration, not a quality metric. |
| Control runtime | 3.815 s | Wall-clock time for this recorded control. |
| Watermarked / ProPainter / DiffuEraser / predicted mask / IoU | **Not applicable** | Missing by design because a faithful VideoShield watermark was never embedded in this MOSE clip. |

The original and raw mask below are the shared control context. They are not VideoShield outputs.

Direct link: [VideoShield availability/control JSON](../../evidence/mose-task1/review50/media/0ab75563/methods/videoshield/availability.json)


In [ ]:
show_videos([('Shared original (control input)', 'media/0ab75563/original.mp4', 'Unwatermarked MOSE source used only for the VideoShield negative control.'), ('Shared raw mask (context)', 'media/0ab75563/gt_mask.mp4', 'Selected removal region shown for context; no VideoShield attack was performed.'), ('VideoShield watermarked/attacked outputs', None, 'Authentic VideoShield embedding requires diffusion generation, so no faithful MOSE watermarked, ProPainter, or DiffuEraser videos exist.')])


In [ ]:
show_metadata(['media/0ab75563/methods/videoshield/availability.json'])


## 5. Cross-method reading of this sample

| Method | What was actually measured | ProPainter result | DiffuEraser result | Correct conclusion |
| --- | --- | ---: | ---: | --- |
| VideoSeal | Derived spatial localization against remover-specific effective GT | Global IoU 0.0000 | Global IoU 0.0000 | The documented crop-grid proxy did not overlap the removed region. |
| Safe-Sora | Graphical-watermark recovery fidelity | PSNR 7.8723 dB | PSNR 8.3995 dB | Both attacks substantially degraded graphical recovery; no spatial IoU is available. |
| VideoShield | Unwatermarked negative-control bit accuracy | N/A | N/A | Accuracy 0.4805 is a control near chance, not an embedding/removal result. |

### What can and cannot be compared

- You **can** compare ProPainter and DiffuEraser within VideoSeal using global IoU because they use the same derived-localization procedure and attack-specific effective GT masks.
- You **can** compare the two Safe-Sora recovery PSNR values because both reconstruct the same graphical watermark.
- You **cannot** compare Safe-Sora PSNR numerically with VideoSeal IoU; they measure different outputs.
- You **cannot** rank VideoShield against the other two on this MOSE sample. Its negative control is included to document the capability boundary, not to fill missing experiment cells with misleading values.
- None of these watermark metrics alone determines which inpainted video looks best. That remains a visual removal-quality judgment supported by the baseline and attacked videos above.


## 6. Every committed video for this sample

This table is generated from the repository tree. It is the complete MP4 inventory for this sample, including shared baseline media and all compatible watermark/removal outputs.

| Group | Video | What it shows | Size |
| --- | --- | --- | ---: |
| baseline | [diffueraser.mp4](../../evidence/mose-task1/review50/media/0ab75563/diffueraser.mp4) | Object-removal output produced from the unwatermarked source clip. | 1.35 MiB |
| baseline | [gt_mask.mp4](../../evidence/mose-task1/review50/media/0ab75563/gt_mask.mp4) | Raw selected-object mask; white pixels identify the object chosen for removal. | 0.02 MiB |
| safe-sora | [diffueraser.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/diffueraser.mp4) | DiffuEraser attack/removal output for safe-sora. | 0.12 MiB |
| safe-sora | [gt-mask.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/gt-mask.mp4) | Safe-Sora-aligned 16-frame object mask. | 0.00 MiB |
| safe-sora | [propainter.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/propainter.mp4) | ProPainter attack/removal output for safe-sora. | 0.15 MiB |
| safe-sora | [sampled-original.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/sampled-original.mp4) | The 16 frames sampled and resized for Safe-Sora's released protocol. | 0.13 MiB |
| safe-sora | [watermarked.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/watermarked.mp4) | Watermarked input produced by safe-sora before either removal attack. | 0.14 MiB |
| videoseal | [diffueraser.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/diffueraser.mp4) | DiffuEraser attack/removal output for videoseal. | 1.73 MiB |
| videoseal | [effective-gt-mask-diffueraser.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/effective-gt-mask-diffueraser.mp4) | Ground-truth region after DiffuEraser's actual mask preprocessing. | 0.03 MiB |
| videoseal | [effective-gt-mask-propainter.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/effective-gt-mask-propainter.mp4) | Ground-truth region after ProPainter's actual mask preprocessing. | 0.03 MiB |
| videoseal | [pred-mask-diffueraser.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/pred-mask-diffueraser.mp4) | VideoSeal's derived patch-grid predicted tamper mask after DiffuEraser. | 0.02 MiB |
| videoseal | [pred-mask-propainter.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/pred-mask-propainter.mp4) | VideoSeal's derived patch-grid predicted tamper mask after ProPainter. | 0.02 MiB |
| videoseal | [propainter.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/propainter.mp4) | ProPainter attack/removal output for videoseal. | 1.70 MiB |
| videoseal | [watermarked.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/watermarked.mp4) | Watermarked input produced by videoseal before either removal attack. | 1.45 MiB |
| baseline | [original.mp4](../../evidence/mose-task1/review50/media/0ab75563/original.mp4) | Full MOSE source clip shared by the baseline and compatible watermark methods. | 1.54 MiB |
| baseline | [propainter.mp4](../../evidence/mose-task1/review50/media/0ab75563/propainter.mp4) | Object-removal output produced from the unwatermarked source clip. | 2.03 MiB |


## 7. Complete artifact inventory

Every committed file for this sample is linked below, including videos, recovery images, metrics, settings, adapters, and integrity hashes.

| Artifact | Type | Description |
| --- | --- | --- |
| [diffueraser.mp4](../../evidence/mose-task1/review50/media/0ab75563/diffueraser.mp4) | MP4 | Object-removal output produced from the unwatermarked source clip. |
| [gt_mask.mp4](../../evidence/mose-task1/review50/media/0ab75563/gt_mask.mp4) | MP4 | Raw selected-object mask; white pixels identify the object chosen for removal. |
| [diffueraser.json](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/diffueraser.json) | JSON | safe-sora + DiffuEraser run settings, runtime, output properties, and checksum. |
| [diffueraser.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/diffueraser.mp4) | MP4 | DiffuEraser attack/removal output for safe-sora. |
| [embed.json](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/embed.json) | JSON | Embedding settings, runtime, media properties, checksums, and fidelity metrics. |
| [gt-mask.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/gt-mask.mp4) | MP4 | Safe-Sora-aligned 16-frame object mask. |
| [propainter.json](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/propainter.json) | JSON | safe-sora + ProPainter run settings, runtime, output properties, and checksum. |
| [propainter.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/propainter.mp4) | MP4 | ProPainter attack/removal output for safe-sora. |
| [recovery-diffueraser.png](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/recovery-diffueraser.png) | PNG | Safe-Sora graphical-watermark recovery after DiffuEraser. |
| [recovery-propainter.png](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/recovery-propainter.png) | PNG | Safe-Sora graphical-watermark recovery after ProPainter. |
| [recovery.json](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/recovery.json) | JSON | Safe-Sora recovery metrics and recovery-image checksums. |
| [sampled-original.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/sampled-original.mp4) | MP4 | The 16 frames sampled and resized for Safe-Sora's released protocol. |
| [watermarked.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/safe-sora/watermarked.mp4) | MP4 | Watermarked input produced by safe-sora before either removal attack. |
| [diffueraser.json](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/diffueraser.json) | JSON | videoseal + DiffuEraser run settings, runtime, output properties, and checksum. |
| [diffueraser.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/diffueraser.mp4) | MP4 | DiffuEraser attack/removal output for videoseal. |
| [effective-gt-mask-diffueraser.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/effective-gt-mask-diffueraser.mp4) | MP4 | Ground-truth region after DiffuEraser's actual mask preprocessing. |
| [effective-gt-mask-propainter.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/effective-gt-mask-propainter.mp4) | MP4 | Ground-truth region after ProPainter's actual mask preprocessing. |
| [embed.json](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/embed.json) | JSON | Embedding settings, runtime, media properties, checksums, and fidelity metrics. |
| [localization.json](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/localization.json) | JSON | VideoSeal localization mode, masks, IoU values, and per-frame measurements. |
| [pred-mask-diffueraser.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/pred-mask-diffueraser.mp4) | MP4 | VideoSeal's derived patch-grid predicted tamper mask after DiffuEraser. |
| [pred-mask-propainter.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/pred-mask-propainter.mp4) | MP4 | VideoSeal's derived patch-grid predicted tamper mask after ProPainter. |
| [propainter.json](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/propainter.json) | JSON | videoseal + ProPainter run settings, runtime, output properties, and checksum. |
| [propainter.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/propainter.mp4) | MP4 | ProPainter attack/removal output for videoseal. |
| [watermarked.mp4](../../evidence/mose-task1/review50/media/0ab75563/methods/videoseal/watermarked.mp4) | MP4 | Watermarked input produced by videoseal before either removal attack. |
| [availability.json](../../evidence/mose-task1/review50/media/0ab75563/methods/videoshield/availability.json) | JSON | VideoShield capability/availability record and negative-control result. |
| [original.mp4](../../evidence/mose-task1/review50/media/0ab75563/original.mp4) | MP4 | Full MOSE source clip shared by the baseline and compatible watermark methods. |
| [propainter.mp4](../../evidence/mose-task1/review50/media/0ab75563/propainter.mp4) | MP4 | Object-removal output produced from the unwatermarked source clip. |


## 8. Validation and provenance

- The final bundle validator decoded all committed videos, checked frame count, dimensions and FPS, rejected empty effective GT masks, enforced explicit `not applicable` fields, and independently recomputed every displayed VideoSeal global IoU.
- Aggregate and per-file validation: [validation.json](../../evidence/mose-task1/review50/validation.json)
- Aggregate method results: [watermark-summary.json](../../evidence/mose-task1/review50/watermark-summary.json)
- Exact experimental contract: [PROTOCOL.md](../../evidence/mose-task1/PROTOCOL.md)
- Batch result summary: [README.md](../../evidence/mose-task1/README.md)

This notebook adds no new experimental claims. It reorganizes the already committed and validated evidence for sample `0ab75563` into one readable case study.
